# Relational Graph Convolutional Network (R-GCN)

Implementation following:
> *Modeling Relational Data with Graph Convolutional Networks*  
> Schlichtkrull, Kipf, Bloem, Berg, Titov, Welling (2017)  
> https://arxiv.org/abs/1703.06103

**Structure:**
1. Shared data structures
2. R-GCN from scratch (every line maps to a paper equation)
3. R-GCN using PyTorch Geometric
4. Wikidata (tkgl-smallpedia) experiments
5. Mystery movie graph experiments (placeholder)

---
## 0. Imports & Setup

In [1]:
import sys
%pip install py-tgb torch-geometric

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Project folders ──────────────────────────────────────────────────────────
# Keep downloaded data and generated checkpoints in local folders next to the
# notebook. If you want these somewhere else, edit PROJECT_DIR below.
import os
PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "datasets")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


Data directory: /Users/abhi/ml/290Final/R-GCN/datasets
Checkpoint directory: /Users/abhi/ml/290Final/R-GCN/checkpoints


In [3]:
import os, json, pickle, time, zipfile
import requests
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)

if torch.cuda.is_available():
    print(f"CUDA device  : {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch  : 2.11.0
Device   : mps


---
## 1. Shared Data Structure

A single `RelationalGraph` dataclass holds everything both datasets need.
Dataset-specific loaders (Sections 4 & 5) populate this and pass it to the model.

In [4]:
@dataclass
class RelationalGraph:
    """
    A dataset-agnostic container for a relational graph.

    All edge tensors have shape (E, 3): [src, dst, rel_type]
    Node features are optional; if absent the model uses a learned embedding.
    """
    name: str                          # human-readable name e.g. "tkgl-smallpedia"
    num_nodes: int                     # total number of nodes
    num_relations: int                 # total number of relation types

    # Edge splits — each is a LongTensor of shape (E_split, 3): [src, dst, rel]
    train_edges: torch.Tensor
    val_edges:   torch.Tensor
    test_edges:  torch.Tensor

    # Optional: pre-computed node feature matrix (num_nodes, feat_dim)
    # If None the model will use a learned embedding table instead
    node_features: Optional[torch.Tensor] = None

    # Optional: human-readable label maps for interpretability
    node_labels: dict = field(default_factory=dict)  # int → string
    rel_labels:  dict = field(default_factory=dict)  # int → string

    def summary(self):
        print(f"RelationalGraph: {self.name}")
        print(f"  Nodes         : {self.num_nodes:,}")
        print(f"  Relation types: {self.num_relations:,}")
        print(f"  Train edges   : {len(self.train_edges):,}")
        print(f"  Val edges     : {len(self.val_edges):,}")
        print(f"  Test edges    : {len(self.test_edges):,}")
        print(f"  Node features : {'yes' if self.node_features is not None else 'no (will use learned embeddings)'}")

---
## 2. R-GCN From Scratch

### 2.1 The Core Equation

Each R-GCN layer computes new node representations by aggregating messages
from typed neighbourhoods (Equation 2 in the paper):

$$h_i^{(l+1)} = \sigma\!\left(W_0^{(l)}\, h_i^{(l)} \;+\; \sum_{r \in \mathcal{R}} \sum_{j \in \mathcal{N}_i^r} \frac{1}{|\mathcal{N}_i^r|}\, W_r^{(l)}\, h_j^{(l)}\right)$$

- $h_i^{(l)}$ is the representation of node $i$ at layer $l$
- $\mathcal{N}_i^r$ is the set of neighbours of $i$ under relation $r$
- $W_0^{(l)}$ is a self-loop weight (the node keeps its own information)
- $W_r^{(l)}$ is a relation-specific weight matrix

### 2.2 Basis Decomposition

With 566 relation types, having a separate $W_r$ for each $r$ is too many
parameters. Basis decomposition (Equation 3) instead writes each $W_r$ as a
weighted sum of $B$ shared basis matrices $\{V_b\}$:

$$W_r^{(l)} = \sum_{b=1}^{B} a_{rb}^{(l)}\, V_b^{(l)}$$

The bases $V_b$ are shared across all relations; only the scalar coefficients
$a_{rb}$ are relation-specific. This reduces parameters from
$|\mathcal{R}| \times d_{in} \times d_{out}$ to
$B \times d_{in} \times d_{out} + |\mathcal{R}| \times B$.

In [5]:
class RGCNLayerScratch(nn.Module):
    """
    Single R-GCN layer with basis decomposition, implemented from scratch.

    Maps:  (num_nodes, in_dim)  →  (num_nodes, out_dim)

    Parameters
    ----------
    in_dim       : input feature dimension
    out_dim      : output feature dimension
    num_relations: number of distinct edge types  (|R| in the paper)
    num_bases    : number of basis matrices B  (paper recommends 10–100)
    """
    def __init__(self, in_dim: int, out_dim: int,
                 num_relations: int, num_bases: int):
        super().__init__()
        self.in_dim        = in_dim
        self.out_dim       = out_dim
        self.num_relations = num_relations
        self.num_bases     = num_bases

        # --- Basis decomposition parameters (Eq. 3) -------------------------
        # V_b: the B shared basis matrices, each (in_dim, out_dim)
        self.bases = nn.Parameter(
            torch.Tensor(num_bases, in_dim, out_dim)
        )
        # a_rb: per-relation coefficients over the bases, shape (num_relations, num_bases)
        self.coefficients = nn.Parameter(
            torch.Tensor(num_relations, num_bases)
        )

        # --- Self-loop weight W_0 (Eq. 2) -----------------------------------
        self.self_loop = nn.Parameter(
            torch.Tensor(in_dim, out_dim)
        )

        self._reset_parameters()

    def _reset_parameters(self):
        # Glorot (Xavier) uniform initialisation — same as the paper
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.self_loop)
        nn.init.xavier_uniform_(self.coefficients)

    def forward(self,
                x: torch.Tensor,          # (num_nodes, in_dim)
                edge_index: torch.Tensor, # (2, E)  — row 0 = src, row 1 = dst
                edge_type: torch.Tensor   # (E,)    — relation index per edge
               ) -> torch.Tensor:         # (num_nodes, out_dim)

        num_nodes = x.size(0)

        # --- Step 1: compute W_r for each relation via basis decomposition --
        # W_r = sum_b a_rb * V_b  →  shape: (num_relations, in_dim, out_dim)
        W = torch.einsum('rb,bio->rio', self.coefficients, self.bases)

        # --- Step 2: aggregate neighbour messages for each relation ---------
        src, dst = edge_index[0], edge_index[1]

        # Degree normalisation: compute inverse degree once
        deg = torch.zeros(num_nodes, device=x.device)
        deg.scatter_add_(0, dst, torch.ones(len(dst), device=x.device))
        deg_inv = (1.0 / deg.clamp(min=1.0))  # avoid division by zero

        agg = torch.zeros(num_nodes, self.out_dim, device=x.device)

        # Memory-efficient aggregation: loop over relations instead of a massive BMM
        for r in range(self.num_relations):
            edge_mask = (edge_type == r)
            if not edge_mask.any():
                continue

            r_src = src[edge_mask]
            r_dst = dst[edge_mask]

            # Transform source node features with the specific relation weight
            # x[r_src] is (E_r, in_dim), W[r] is (in_dim, out_dim)
            msg = x[r_src] @ W[r]

            # Apply normalization (E_r,)
            msg = msg * deg_inv[r_dst].unsqueeze(1)

            # Scatter add messages into destination nodes
            agg.scatter_add_(0, r_dst.unsqueeze(1).expand(-1, self.out_dim), msg)

        # --- Step 3: add self-loop (W_0 * h_i) and activate ----------------
        out = agg + x @ self.self_loop  # (num_nodes, out_dim)
        return out

In [6]:
class RGCNScratch(nn.Module):
    """
    Multi-layer R-GCN encoder (from scratch).

    Produces node embeddings from a relational graph.
    If the graph has no node features, a learned embedding table is used
    as the input layer (common practice for knowledge graphs).

    Parameters
    ----------
    num_nodes    : number of nodes
    num_relations: number of relation types
    hidden_dim   : dimension of hidden (and output) representations
    num_layers   : number of R-GCN layers (paper uses 2)
    num_bases    : number of basis matrices for decomposition
    feat_dim     : input feature dimension; if None uses a learned embedding
    dropout      : dropout rate applied between layers
    """
    def __init__(self,
                 num_nodes: int,
                 num_relations: int,
                 hidden_dim: int   = 64,
                 num_layers: int   = 2,
                 num_bases: int    = 30,
                 feat_dim: Optional[int] = None,
                 dropout: float    = 0.1):
        super().__init__()
        self.dropout = dropout

        # Input: either a learned embedding table or a linear projection
        if feat_dim is None:
            # No node features: learn a (num_nodes, hidden_dim) embedding table
            self.embedding = nn.Embedding(num_nodes, hidden_dim)
            in_dim = hidden_dim
        else:
            # Project input features to hidden_dim
            self.embedding = None
            self.input_proj = nn.Linear(feat_dim, hidden_dim)
            in_dim = hidden_dim

        # Stack of R-GCN layers
        self.layers = nn.ModuleList([
            RGCNLayerScratch(in_dim, hidden_dim, num_relations, num_bases)
            for _ in range(num_layers)
        ])

    def forward(self,
                edge_index: torch.Tensor,  # (2, E)
                edge_type: torch.Tensor,   # (E,)
                node_features: Optional[torch.Tensor] = None,  # (N, feat_dim)
                num_nodes: Optional[int] = None
               ) -> torch.Tensor:          # (N, hidden_dim)

        # Get initial node representations
        if self.embedding is not None:
            N = num_nodes or (edge_index.max().item() + 1)
            x = self.embedding(torch.arange(N, device=edge_index.device))
        else:
            x = F.relu(self.input_proj(node_features))

        # Pass through R-GCN layers with ReLU + dropout between them
        for i, layer in enumerate(self.layers):
            x = layer(x, edge_index, edge_type)
            if i < len(self.layers) - 1:  # no activation on the last layer
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        return x  # (num_nodes, hidden_dim) — final node embeddings

### 2.3 Link Prediction Decoder

For link prediction the paper uses the **DistMult** scoring function (Section 4).
Given subject $s$, relation $r$, object $o$, the score is:

$$f(s, r, o) = h_s^\top \, \text{diag}(R_r) \, h_o$$

where $R_r$ is a learned diagonal matrix per relation type.
Higher score = more likely to be a true fact.

In [7]:
class DistMultDecoder(nn.Module):
    """
    DistMult link prediction decoder (Section 4 of the paper).

    Scores a triple (s, r, o) as:  h_s . diag(R_r) . h_o
    """
    def __init__(self, num_relations: int, hidden_dim: int):
        super().__init__()
        # R_r: one diagonal vector per relation, shape (num_relations, hidden_dim)
        self.relation_emb = nn.Embedding(num_relations, hidden_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self,
                node_emb: torch.Tensor,   # (num_nodes, hidden_dim)
                src: torch.Tensor,        # (E,) source indices
                dst: torch.Tensor,        # (E,) destination indices
                rel: torch.Tensor         # (E,) relation indices
               ) -> torch.Tensor:         # (E,) scores

        h_s = node_emb[src]               # (E, hidden_dim)
        h_o = node_emb[dst]               # (E, hidden_dim)
        r   = self.relation_emb(rel)      # (E, hidden_dim)  — diagonal of R_r

        # Element-wise product then sum = h_s . diag(R_r) . h_o
        scores = (h_s * r * h_o).sum(dim=-1)  # (E,)
        return scores


class RGCNLinkPredictor(nn.Module):
    """
    Full model: R-GCN encoder + DistMult decoder.
    This is the complete architecture from Section 4 of the paper.
    """
    def __init__(self, num_nodes, num_relations, hidden_dim=64,
                 num_layers=2, num_bases=30, feat_dim=None, dropout=0.1):
        super().__init__()
        self.encoder = RGCNScratch(
            num_nodes, num_relations, hidden_dim,
            num_layers, num_bases, feat_dim, dropout
        )
        self.decoder = DistMultDecoder(num_relations, hidden_dim)

    def forward(self,
                # Graph structure (used by encoder)
                edge_index: torch.Tensor,
                edge_type: torch.Tensor,
                # Triples to score (used by decoder)
                src: torch.Tensor,
                dst: torch.Tensor,
                rel: torch.Tensor,
                node_features: Optional[torch.Tensor] = None,
                num_nodes: Optional[int] = None
               ) -> torch.Tensor:

        node_emb = self.encoder(edge_index, edge_type, node_features, num_nodes)
        scores   = self.decoder(node_emb, src, dst, rel)
        return scores

---
## 3. R-GCN Using PyTorch Geometric

PyG's `RGCNConv` handles the message-passing mechanics internally.
We still write our own basis decomposition flag, decoder, and training loop —
only the aggregation step is delegated to PyG.

In [8]:
try:
    from torch_geometric.nn import RGCNConv
    PYG_AVAILABLE = True
    print("PyTorch Geometric available — PyG model will work.")
except ImportError:
    PYG_AVAILABLE = False
    print("PyTorch Geometric not installed. Install with:")
    print("  pip install torch_geometric")
    print("PyG model cells will be skipped.")

PyTorch Geometric available — PyG model will work.


In [9]:
if PYG_AVAILABLE:
    class RGCNPyG(nn.Module):
        """
        R-GCN encoder built with PyTorch Geometric's RGCNConv.

        Functionally identical to RGCNScratch above.
        PyG's RGCNConv accepts num_bases directly and handles
        the basis decomposition internally.
        """
        def __init__(self, num_nodes, num_relations, hidden_dim=64,
                     num_layers=2, num_bases=30, feat_dim=None, dropout=0.1):
            super().__init__()
            self.dropout = dropout

            if feat_dim is None:
                self.embedding = nn.Embedding(num_nodes, hidden_dim)
                in_dim = hidden_dim
            else:
                self.embedding = None
                self.input_proj = nn.Linear(feat_dim, hidden_dim)
                in_dim = hidden_dim

            # PyG RGCNConv: num_bases triggers basis decomposition
            self.layers = nn.ModuleList([
                RGCNConv(in_dim, hidden_dim,
                         num_relations=num_relations,
                         num_bases=num_bases)
                for _ in range(num_layers)
            ])

        def forward(self, edge_index, edge_type,
                    node_features=None, num_nodes=None):
            if self.embedding is not None:
                N = num_nodes or (edge_index.max().item() + 1)
                x = self.embedding(torch.arange(N, device=edge_index.device))
            else:
                x = F.relu(self.input_proj(node_features))

            for i, layer in enumerate(self.layers):
                x = layer(x, edge_index, edge_type)
                if i < len(self.layers) - 1:
                    x = F.relu(x)
                    x = F.dropout(x, p=self.dropout, training=self.training)
            return x

    class RGCNLinkPredictorPyG(nn.Module):
        """Full model: PyG R-GCN encoder + DistMult decoder."""
        def __init__(self, num_nodes, num_relations, hidden_dim=64,
                     num_layers=2, num_bases=30, feat_dim=None, dropout=0.1):
            super().__init__()
            self.encoder = RGCNPyG(
                num_nodes, num_relations, hidden_dim,
                num_layers, num_bases, feat_dim, dropout
            )
            self.decoder = DistMultDecoder(num_relations, hidden_dim)

        def forward(self, edge_index, edge_type, src, dst, rel,
                    node_features=None, num_nodes=None):
            node_emb = self.encoder(edge_index, edge_type, node_features, num_nodes)
            return self.decoder(node_emb, src, dst, rel)

---
## 4. Shared Training & Evaluation

Both models share the same training loop.
We use the negative sampling + binary cross-entropy loss from Section 4 of the paper:
for each positive triple $(s, r, o)$ we sample a random negative triple
$(s, r, o')$ by corrupting the object node.

In [10]:
def sample_negatives(pos_edges: torch.Tensor,
                     num_nodes: int,
                     neg_ratio: int = 1) -> torch.Tensor:
    """
    For each positive edge (s, r, o), sample neg_ratio negative edges
    by randomly corrupting the destination node.

    Returns a tensor of shape (E * neg_ratio, 3).
    """
    src = pos_edges[:, 0].repeat(neg_ratio)
    rel = pos_edges[:, 2].repeat(neg_ratio)
    dst = torch.randint(0, num_nodes, (len(src),), device=pos_edges.device)
    return torch.stack([src, dst, rel], dim=1)


def train_epoch(model: nn.Module,
                graph: RelationalGraph,
                optimizer: torch.optim.Optimizer,
                edge_index: torch.Tensor,  # (2, E_train) — full training graph
                edge_type: torch.Tensor,   # (E_train,)
                batch_size: int = 4096,
                neg_ratio: int = 1
               ) -> float:
    model.train()
    train_edges = graph.train_edges.to(DEVICE)
    total_loss  = 0.0
    num_batches = 0

    # Shuffle training edges
    perm = torch.randperm(len(train_edges))
    train_edges = train_edges[perm]

    for start in range(0, len(train_edges), batch_size):
        pos = train_edges[start : start + batch_size]
        neg = sample_negatives(pos, graph.num_nodes, neg_ratio)

        # Concatenate positive and negative edges
        all_edges = torch.cat([pos, neg], dim=0)
        labels = torch.cat([
            torch.ones(len(pos)),
            torch.zeros(len(neg))
        ]).to(DEVICE)

        scores = model(
            edge_index, edge_type,
            all_edges[:, 0], all_edges[:, 1], all_edges[:, 2],
            node_features=graph.node_features,
            num_nodes=graph.num_nodes
        )

        loss = F.binary_cross_entropy_with_logits(scores, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        num_batches += 1

    return total_loss / num_batches


@torch.no_grad()
def evaluate(model: nn.Module,
             graph: RelationalGraph,
             split_edges: torch.Tensor,   # val or test edges
             edge_index: torch.Tensor,
             edge_type: torch.Tensor,
             num_negatives: int = 50      # negatives per positive for ranking
            ) -> dict:
    """
    Evaluate using Mean Reciprocal Rank (MRR) and Hits@K.
    For each positive triple we rank it against num_negatives corrupted triples.
    """
    model.eval()
    split_edges = split_edges.to(DEVICE)

    # Get all node embeddings once
    node_emb = model.encoder(
        edge_index, edge_type,
        node_features=graph.node_features,
        num_nodes=graph.num_nodes
    )

    ranks = []
    for pos in split_edges:
        s, o, r = pos[0], pos[1], pos[2]

        # Score the positive triple
        pos_score = model.decoder(
            node_emb,
            s.unsqueeze(0), o.unsqueeze(0), r.unsqueeze(0)
        ).item()

        # Sample and score negatives
        neg_dst = torch.randint(0, graph.num_nodes, (num_negatives,), device=DEVICE)
        neg_scores = model.decoder(
            node_emb,
            s.expand(num_negatives), neg_dst, r.expand(num_negatives)
        )

        # Rank = number of negatives that score higher than the positive + 1
        rank = (neg_scores >= pos_score).sum().item() + 1
        ranks.append(rank)

    ranks = np.array(ranks)
    return {
        'mrr':    float(np.mean(1.0 / ranks)),
        'hits@1': float(np.mean(ranks <= 1)),
        'hits@3': float(np.mean(ranks <= 3)),
        'hits@10': float(np.mean(ranks <= 10)),
    }


def train(model: nn.Module,
          graph: RelationalGraph,
          num_epochs: int   = 50,
          lr: float         = 1e-3,
          batch_size: int   = 4096,
          eval_every: int   = 5,
          neg_ratio: int    = 1
         ) -> dict:
    """Full training loop, shared across both datasets and both model variants."""

    model = model.to(DEVICE)
    optimizer = Adam(model.parameters(), lr=lr)

    # Move node features to device if present
    if graph.node_features is not None:
        graph.node_features = graph.node_features.to(DEVICE)

    # Build the graph structure tensors for the encoder
    # (encoder always sees the full training graph)
    train_e    = graph.train_edges.to(DEVICE)
    edge_index = train_e[:, :2].t().contiguous()  # (2, E_train)
    edge_type  = train_e[:, 2]                    # (E_train,)

    history = {'train_loss': [], 'val_mrr': [], 'val_hits10': []}

    for epoch in range(1, num_epochs + 1):
        loss = train_epoch(model, graph, optimizer, edge_index, edge_type,
                           batch_size, neg_ratio)
        history['train_loss'].append(loss)

        if epoch % eval_every == 0:
            metrics = evaluate(model, graph, graph.val_edges, edge_index, edge_type)
            history['val_mrr'].append(metrics['mrr'])
            history['val_hits10'].append(metrics['hits@10'])
            print(f"Epoch {epoch:3d} | Loss {loss:.4f} | "
                  f"Val MRR {metrics['mrr']:.4f} | "
                  f"Hits@10 {metrics['hits@10']:.4f}")
        else:
            print(f"Epoch {epoch:3d} | Loss {loss:.4f}")

    return history


def plot_history(history: dict, title: str):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)
    ax1.plot(history['train_loss'])
    ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
    ax2.plot(history['val_mrr'],    label='MRR')
    ax2.plot(history['val_hits10'], label='Hits@10')
    ax2.set_title('Validation Metrics'); ax2.set_xlabel('Eval step')
    ax2.legend()
    plt.tight_layout(); plt.show()

---
## 5. Wikidata Experiments (tkgl-smallpedia)

In [11]:
# ── Wikidata label fetching ───────────────────────────────────────────────────
TGB_DATASET_NAME = "tkgl-smallpedia"
TGB_DATASET_DIR = os.path.join(DATA_DIR, "tkgl_smallpedia")
LABEL_CACHE_PATH = os.path.join(DATA_DIR, "wikidata_labels_cache.json")
HEADERS = {"User-Agent": "rgcn-project/1.0 (university research)"}

def save_label_cache(label_cache: dict) -> None:
    tmp_path = LABEL_CACHE_PATH + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(label_cache, f)
    os.replace(tmp_path, LABEL_CACHE_PATH)


def ensure_local_tgb_dataset(root: str = DATA_DIR) -> str:
    """Download tkgl-smallpedia into the local DATA_DIR if needed."""
    from tgb.utils.info import DATA_URL_DICT

    root = os.path.abspath(root)
    dataset_dir = os.path.join(root, "tkgl_smallpedia")
    edgelist_path = os.path.join(dataset_dir, f"{TGB_DATASET_NAME}_edgelist.csv")

    if os.path.exists(edgelist_path):
        return dataset_dir

    os.makedirs(dataset_dir, exist_ok=True)
    zip_path = os.path.join(dataset_dir, f"{TGB_DATASET_NAME}.zip")
    url = DATA_URL_DICT[TGB_DATASET_NAME]

    print(f"Downloading {TGB_DATASET_NAME} into {dataset_dir}...")
    with requests.get(url, headers=HEADERS, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(dataset_dir)
    os.remove(zip_path)

    if not os.path.exists(edgelist_path):
        raise FileNotFoundError(
            f"Download completed but {edgelist_path} was not found."
        )

    return dataset_dir


def fetch_wikidata_labels(
    ids: list,
    batch_size: int = 50,
    seed_cache: dict | None = None,
    save_every: int = 25,
    pause_seconds: float = 1.0,
    cooldown_every: int = 200,
    cooldown_seconds: float = 20.0,
) -> dict:
    label_cache = dict(seed_cache or {})
    if not ids:
        return label_cache

    total_batches = (len(ids) + batch_size - 1) // batch_size
    session = requests.Session()
    session.headers.update(HEADERS)

    for batch_num, start in enumerate(range(0, len(ids), batch_size), start=1):
        batch = ids[start : start + batch_size]

        if batch_num > 1 and (batch_num - 1) % cooldown_every == 0:
            print(
                f"Cooling down for {cooldown_seconds:.0f}s before batch "
                f"{batch_num}/{total_batches}..."
            )
            time.sleep(cooldown_seconds)

        for attempt in range(5):
            try:
                resp = session.get(
                    "https://www.wikidata.org/w/api.php",
                    params={
                        "action": "wbgetentities",
                        "ids": "|".join(batch),
                        "props": "labels",
                        "languages": "en",
                        "format": "json",
                    },
                    timeout=30,
                )

                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = max(float(retry_after) if retry_after else 0.0, min(120.0, 5.0 * (2 ** attempt)))
                    print(
                        f"  Batch {batch_num}/{total_batches} was rate-limited; "
                        f"waiting {wait:.0f}s before retrying..."
                    )
                    time.sleep(wait)
                    continue

                resp.raise_for_status()
                entities = resp.json().get("entities", {})
                for qid in batch:
                    info = entities.get(qid, {})
                    label_cache[qid] = info.get("labels", {}).get("en", {}).get("value", qid)
                break
            except (requests.RequestException, ValueError) as e:
                wait = min(120.0, 5.0 * (2 ** attempt))
                print(
                    f"  Batch {batch_num}/{total_batches} attempt {attempt+1} failed ({e}); "
                    f"retrying in {wait:.0f}s..."
                )
                time.sleep(wait)
        else:
            print(f"  Batch {batch_num}/{total_batches} failed repeatedly; keeping raw IDs.")
            for qid in batch:
                label_cache.setdefault(qid, qid)

        if batch_num % save_every == 0 or batch_num == total_batches:
            save_label_cache(label_cache)
            print(
                f"Saved {len(label_cache):,} labels after batch "
                f"{batch_num}/{total_batches}."
            )

        time.sleep(pause_seconds)

    return label_cache


def load_wikidata(root: str = DATA_DIR) -> RelationalGraph:
    """
    Load tkgl-smallpedia via TGB and wrap it in a RelationalGraph.

    - The raw TGB files are downloaded into the local DATA_DIR if needed.
    - We point py-tgb at that local folder so the notebook stays self-contained.
    - English labels are fetched from the Wikidata API on first run and
      cached progressively in DATA_DIR so long runs can resume safely.
    """
    import tgb.linkproppred.dataset as tgb_dataset_module
    from tgb.linkproppred.dataset import LinkPropPredDataset

    root = os.path.abspath(root)
    ensure_local_tgb_dataset(root)

    original_proj_dir = tgb_dataset_module.PROJ_DIR
    tgb_dataset_module.PROJ_DIR = ""
    try:
        dataset = LinkPropPredDataset(
            name=TGB_DATASET_NAME,
            root=root,
            preprocess=True,
            download=False,
        )
    finally:
        tgb_dataset_module.PROJ_DIR = original_proj_dir

    data = dataset.full_data

    src   = torch.from_numpy(data["sources"].astype(np.int64))
    dst   = torch.from_numpy(data["destinations"].astype(np.int64))
    rel   = torch.from_numpy(data["edge_type"].astype(np.int64))
    edges = torch.stack([src, dst, rel], dim=1)

    train_mask = torch.from_numpy(dataset.train_mask)
    val_mask   = torch.from_numpy(dataset.val_mask)
    test_mask  = torch.from_numpy(dataset.test_mask)

    tgb_dir = dataset.root
    print(f"Using TGB files from: {tgb_dir}")

    with open(os.path.join(tgb_dir, "ml_tkgl-smallpedia_nodeid.pkl"), "rb") as f:
        node2int = pickle.load(f)
    int2node = {v: k for k, v in node2int.items()}

    raw         = pd.read_csv(dataset.meta_dict["fname"])
    unique_rels = list(dict.fromkeys(raw["relation_type"]))
    int2rel     = {i: p for i, p in enumerate(unique_rels)}

    # Load or fetch English labels
    if os.path.exists(LABEL_CACHE_PATH):
        print(f"Loading label cache from {LABEL_CACHE_PATH}")
        with open(LABEL_CACHE_PATH) as f:
            label_cache = json.load(f)
    else:
        label_cache = {}

    all_ids = list(set(int2node.values()) | set(int2rel.values()))
    missing = [wid for wid in all_ids if wid not in label_cache]
    if missing:
        print(f"Fetching {len(missing):,} Wikidata labels (first run only — "
              f"results cached to {LABEL_CACHE_PATH})...")
        label_cache = fetch_wikidata_labels(missing, seed_cache=label_cache)
        save_label_cache(label_cache)
        print("Labels cached.")

    node_labels = {i: label_cache.get(q, q) for i, q in int2node.items()}
    rel_labels  = {i: label_cache.get(p, p) for i, p in int2rel.items()}

    return RelationalGraph(
        name          = "tkgl-smallpedia",
        num_nodes     = dataset.num_nodes,
        num_relations = dataset.num_rels,
        train_edges   = edges[train_mask],
        val_edges     = edges[val_mask],
        test_edges    = edges[test_mask],
        node_labels   = node_labels,
        rel_labels    = rel_labels,
    )

wiki_graph = load_wikidata()
wiki_graph.summary()


550377it [00:00, 693891.02it/s]


Using TGB files from: /Users/abhi/ml/290Final/R-GCN/datasets/tkgl_smallpedia
Fetching 47,716 Wikidata labels (first run only — results cached to /Users/abhi/ml/290Final/R-GCN/datasets/wikidata_labels_cache.json)...
Saved 1,250 labels after batch 25/955.
Saved 2,500 labels after batch 50/955.
Saved 3,750 labels after batch 75/955.
Saved 5,000 labels after batch 100/955.
Saved 6,250 labels after batch 125/955.
Saved 7,500 labels after batch 150/955.
Saved 8,750 labels after batch 175/955.
Saved 10,000 labels after batch 200/955.
Cooling down for 20s before batch 201/955...
Saved 11,250 labels after batch 225/955.
Saved 12,500 labels after batch 250/955.
Saved 13,750 labels after batch 275/955.
Saved 15,000 labels after batch 300/955.
Saved 16,250 labels after batch 325/955.
Saved 17,500 labels after batch 350/955.
Saved 18,750 labels after batch 375/955.
Saved 20,000 labels after batch 400/955.
Cooling down for 20s before batch 401/955...
Saved 21,250 labels after batch 425/955.
Saved 22

In [12]:
import os
print('Contents of datasets directory:')
for root_dir, dirs, files in os.walk('datasets'):
    for f in files:
        print(os.path.join(root_dir, f))

Contents of datasets directory:
datasets/wikidata_labels_cache.json
datasets/tkgl_smallpedia/ml_tkgl-smallpedia.pkl
datasets/tkgl_smallpedia/tkgl-smallpedia_edgelist.csv
datasets/tkgl_smallpedia/tkgl-smallpedia_static_edgelist.csv
datasets/tkgl_smallpedia/ml_tkgl-smallpedia_nodeid.pkl
datasets/tkgl_smallpedia/tkgl-smallpedia_test_ns.pkl
datasets/tkgl_smallpedia/ml_tkgl-smallpedia_edge.pkl
datasets/tkgl_smallpedia/tkgl-smallpedia_val_ns.pkl


In [13]:
# ── Checkpoint paths (all relative to CHECKPOINT_DIR) ────────────────────────
SCRATCH_CKPT = os.path.join(CHECKPOINT_DIR, "rgcn_scratch_wikidata.pt")
PYG_CKPT     = os.path.join(CHECKPOINT_DIR, "rgcn_pyg_wikidata.pt")
GRAPH_CKPT   = os.path.join(CHECKPOINT_DIR, "wiki_graph.pkl")

# Persist the graph object so teammates who load from checkpoint skip TGB entirely
if not os.path.exists(GRAPH_CKPT):
    with open(GRAPH_CKPT, "wb") as f:
        pickle.dump(wiki_graph, f)
    print(f"Graph saved to {GRAPH_CKPT}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Scratch model: build ──────────────────────────────────────────────────────
wiki_model_scratch = RGCNLinkPredictor(
    num_nodes     = wiki_graph.num_nodes,
    num_relations = wiki_graph.num_relations,
    hidden_dim    = 64,
    num_layers    = 2,
    num_bases     = 30,
    dropout       = 0.1,
)
print(f"Scratch model parameters: {sum(p.numel() for p in wiki_model_scratch.parameters()):,}")

# ── Scratch model: train or load ──────────────────────────────────────────────
if os.path.exists(SCRATCH_CKPT):
    print(f"Checkpoint found — loading scratch model from {SCRATCH_CKPT}")
    wiki_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
    wiki_model_scratch.to(DEVICE).eval()
    wiki_history_scratch = None
    print("Scratch model ready (training skipped).")
else:
    print("No checkpoint found — training scratch model...")
    wiki_history_scratch = train(
        wiki_model_scratch, wiki_graph,
        num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5,
    )
    torch.save(wiki_model_scratch.state_dict(), SCRATCH_CKPT)
    print(f"Scratch model saved to {SCRATCH_CKPT}")


Graph saved to /Users/abhi/ml/290Final/R-GCN/checkpoints/wiki_graph.pkl
Scratch model parameters: 3,359,848
No checkpoint found — training scratch model...


KeyboardInterrupt: 

In [ ]:
if wiki_history_scratch is not None:
    plot_history(wiki_history_scratch, "Wikidata — R-GCN from scratch")
else:
    print("Model was loaded from checkpoint — no training history to plot.")


In [ ]:
plot_history(wiki_history_scratch, "Wikidata — R-GCN from scratch")

In [ ]:
# --- Final test evaluation --------------------------------------------------
train_e    = wiki_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

test_metrics = evaluate(
    wiki_model_scratch, wiki_graph,
    wiki_graph.test_edges, edge_index, edge_type
)
print("Test results (from-scratch model):")
for k, v in test_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

In [ ]:
if PYG_AVAILABLE:
    wiki_model_pyg = RGCNLinkPredictorPyG(
        num_nodes     = wiki_graph.num_nodes,
        num_relations = wiki_graph.num_relations,
        hidden_dim    = 64,
        num_layers    = 2,
        num_bases     = 30,
        dropout       = 0.1,
    )
    print(f"PyG model parameters: {sum(p.numel() for p in wiki_model_pyg.parameters()):,}")

    if os.path.exists(PYG_CKPT):
        print(f"Checkpoint found — loading PyG model from {PYG_CKPT}")
        wiki_model_pyg.load_state_dict(torch.load(PYG_CKPT, map_location=DEVICE))
        wiki_model_pyg.to(DEVICE).eval()
        wiki_history_pyg = None
        print("PyG model ready (training skipped).")
    else:
        print("No PyG checkpoint found — training...")
        wiki_history_pyg = train(
            wiki_model_pyg, wiki_graph,
            num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5,
        )
        torch.save(wiki_model_pyg.state_dict(), PYG_CKPT)
        print(f"PyG model saved to {PYG_CKPT}")


In [ ]:
if PYG_AVAILABLE:
    if wiki_history_pyg is not None:
        plot_history(wiki_history_pyg, "Wikidata — R-GCN (PyG)")
    else:
        print("PyG model was loaded from checkpoint — no training history to plot.")


In [ ]:
# --- Compare scratch vs PyG side by side -----------------------------------
if PYG_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Wikidata: Scratch vs PyG", fontsize=13)
    for hist, label, ls in [
        (wiki_history_scratch, 'Scratch', '-'),
        (wiki_history_pyg,     'PyG',     '--')
    ]:
        axes[0].plot(hist['train_loss'], ls=ls, label=label)
        axes[1].plot(hist['val_mrr'],    ls=ls, label=f"{label} MRR")
        axes[1].plot(hist['val_hits10'], ls=ls, alpha=0.6, label=f"{label} Hits@10")
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].set_title('Val Metrics'); axes[1].legend()
    plt.tight_layout(); plt.show()

---
## 6. Mystery Movie Graph Experiments

**Placeholder** — fill in `load_mystery_movies()` once you have the dataset.

The function should return a `RelationalGraph` with:
- `num_nodes` = 500
- edges as a `(E, 3)` LongTensor: `[src, dst, rel_type]`
- optionally, `node_features` if your nodes have attributes
- optionally, `node_labels` and `rel_labels` dicts for interpretability

The training and evaluation code below will work unchanged once
`load_mystery_movies()` returns a valid `RelationalGraph`.

In [ ]:
def load_mystery_movies(path: str = "mystery_movies.csv") -> RelationalGraph:
    """
    PLACEHOLDER — replace with your actual data loading logic.

    Expected input format (adjust as needed once you have the data):
        CSV with columns: src, dst, relation_type
        where src/dst are node IDs (integers or strings)
        and relation_type is a string edge label.

    Returns a RelationalGraph ready to pass to train().
    """
    raise NotImplementedError(
        "Replace this placeholder with your actual data loader.\n"
        "See the docstring above for expected format."
    )

    # ── Template for when you have the data ─────────────────────────────────
    # df = pd.read_csv(path)
    #
    # # Encode string node/relation IDs as integers
    # all_nodes = pd.concat([df['src'], df['dst']]).unique()
    # node2int  = {n: i for i, n in enumerate(all_nodes)}
    # rel_types = df['relation_type'].unique()
    # rel2int   = {r: i for i, r in enumerate(rel_types)}
    #
    # edges = torch.tensor([
    #     [node2int[s], node2int[d], rel2int[r]]
    #     for s, d, r in zip(df['src'], df['dst'], df['relation_type'])
    # ], dtype=torch.long)
    #
    # # Simple chronological 70/15/15 split
    # n = len(edges)
    # t1, t2 = int(0.7 * n), int(0.85 * n)
    # train_edges = edges[:t1]
    # val_edges   = edges[t1:t2]
    # test_edges  = edges[t2:]
    #
    # return RelationalGraph(
    #     name          = "mystery-movies",
    #     num_nodes     = len(all_nodes),
    #     num_relations = len(rel_types),
    #     train_edges   = train_edges,
    #     val_edges     = val_edges,
    #     test_edges    = test_edges,
    #     node_labels   = {v: k for k, v in node2int.items()},
    #     rel_labels    = {v: k for k, v in rel2int.items()},
    # )

In [ ]:
# Uncomment once load_mystery_movies() is implemented

# movie_graph = load_mystery_movies("path/to/your/data.csv")
# movie_graph.summary()
#
# movie_model = RGCNLinkPredictor(
#     num_nodes    = movie_graph.num_nodes,
#     num_relations= movie_graph.num_relations,
#     hidden_dim   = 32,    # smaller model for smaller graph
#     num_layers   = 2,
#     num_bases    = 10,    # fewer bases needed with fewer relation types
#     dropout      = 0.2,
# )
#
# movie_history = train(
#     movie_model, movie_graph,
#     num_epochs = 100,    # more epochs for smaller dataset
#     lr         = 1e-3,
#     batch_size = 256,    # smaller batches for smaller graph
#     eval_every = 10,
# )
#
# plot_history(movie_history, "Mystery Movies — R-GCN")

In [ ]:
# Manual checkpoint save (only needed if you want to force-overwrite a checkpoint)
# Under normal use, cells 20 and 24 handle saving automatically after training.
#
# Uncomment and run if you need to save manually:
#
# torch.save(wiki_model_scratch.state_dict(), SCRATCH_CKPT)
# print(f"Scratch model saved to {SCRATCH_CKPT}")
#
# if PYG_AVAILABLE:
#     torch.save(wiki_model_pyg.state_dict(), PYG_CKPT)
#     print(f"PyG model saved to {PYG_CKPT}")


In [ ]:
# Manual checkpoint load — use this if you want to reload weights mid-session
# without re-running cells 20/24 (e.g. after accidentally overwriting a model).
#
# wiki_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
# wiki_model_scratch.to(DEVICE).eval()
# print("Scratch model reloaded.")


In [ ]:
# English labels are now loaded automatically inside load_wikidata() and
# cached to DATA_DIR/wikidata_labels_cache.json.
#
# If you already have a wikidata_labels_cache.json file (e.g. from a teammate),
# copy it into DATA_DIR before running load_wikidata() and the API fetch
# will be skipped entirely:
#
#   import shutil
#   shutil.copy("wikidata_labels_cache.json", LABEL_CACHE_PATH)
#
# Verify labels loaded correctly:
print("Sample node labels:")
print(list(wiki_graph.node_labels.items())[:5])
print("\nSample relation labels:")
print(list(wiki_graph.rel_labels.items())[:5])


In [ ]:
# ── Pre-compute node embeddings for prediction ────────────────────────────────
# Run this after either training or loading the scratch model (cell 20).
# node_emb is reused by predict_object() and predict_relation() below.

train_e    = wiki_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

# Use wiki_model_scratch (or swap to wiki_model_pyg / model if you prefer)
active_model = wiki_model_scratch

with torch.no_grad():
    node_emb = active_model.encoder(
        edge_index, edge_type,
        num_nodes=wiki_graph.num_nodes
    )

print(f"Node embeddings shape: {node_emb.shape}")
model = active_model   # alias used by predict_object / predict_relation


In [ ]:
# Build the graph structure tensors
train_e    = wiki_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()  # (2, E)
edge_type  = train_e[:, 2]                    # (E,)

# Run the encoder once — this is the expensive step
with torch.no_grad():
    node_emb = model.encoder(
        edge_index, edge_type,
        num_nodes=wiki_graph.num_nodes
    )

print(f"Node embeddings shape: {node_emb.shape}")
# Should print: torch.Size([47433, 64])

In [ ]:
def predict_object(subject: str, relation: str, top_k: int = 10):
    """
    Given a subject entity and relation (as English strings),
    return the top_k most likely object entities.
    """
    # Look up integer indices from English labels
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}
    rel2int  = {v: k for k, v in wiki_graph.rel_labels.items()}

    s_idx, r_idx = None, None

    if subject not in node2int:
        print(f"Subject '{subject}' not found in node labels.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        print("  Perhaps try a Wikidata Q-code (e.g., 'Q12345') if you know it.")
        return
    else:
        s_idx = node2int[subject]

    if relation not in rel2int:
        print(f"Relation '{relation}' not found in relation labels.")
        matches = [r for r in rel2int if relation.lower() in r.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        print("  Perhaps try a Wikidata P-code (e.g., 'P123') if you know it.")
        return
    else:
        r_idx = rel2int[relation]

    # Score all possible object nodes
    with torch.no_grad():
        h_s = node_emb[s_idx].unsqueeze(0)                          # (1, hidden_dim)
        r   = model.decoder.relation_emb(
                  torch.tensor([r_idx], device=DEVICE))             # (1, hidden_dim)
        # Broadcast over all nodes: scores = sum(h_s * r * h_o, dim=-1)
        scores = (h_s * r * node_emb).sum(dim=-1)                   # (num_nodes,)

    top_scores, top_indices = scores.topk(top_k)

    print(f"\n  '{subject}'  --[{relation}]-->  ???\n")
    print(f"  {'Rank':<6} {'Entity':<35} {'Score':>8}")
    print("  " + "-" * 52)
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(),
                                             top_scores.tolist()), 1):
        label = wiki_graph.node_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<35} {score:>8.4f}")

In [ ]:
def predict_relation(subject: str, obj: str, top_k: int = 5):
    """
    Given subject and object entities, return the most likely relations.
    """
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}

    s_idx, o_idx = None, None

    if subject not in node2int:
        print(f"Subject '{subject}' not found in node labels.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        print("  Perhaps try a Wikidata Q-code (e.g., 'Q12345') if you know it.")
        return
    else:
        s_idx = node2int[subject]

    if obj not in node2int:
        print(f"Object '{obj}' not found in node labels.")
        matches = [n for n in node2int if obj.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        print("  Perhaps try a Wikidata Q-code (e.g., 'Q12345') if you know it.")
        return
    else:
        o_idx = node2int[obj]

    with torch.no_grad():
        h_s = node_emb[s_idx]   # (hidden_dim,)
        h_o = node_emb[o_idx]   # (hidden_dim,)
        # Score all relation types: sum(h_s * r * h_o) for each r
        all_r  = model.decoder.relation_emb.weight  # (num_relations, hidden_dim)
        scores = (h_s * all_r * h_o).sum(dim=-1)    # (num_relations,)

    top_scores, top_indices = scores.topk(top_k)

    print(f"\n  '{subject}'  --[???]-->  '{obj}'\n")
    print(f"  {'Rank':<6} {'Relation':<35} {'Score':>8}")
    print("  " + "-" * 52)
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(),
                                             top_scores.tolist()), 1):
        label = wiki_graph.rel_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<35} {score:>8.4f}")

In [ ]:
# Predict what awards Lille has received
predict_object("Lille", "award received")

# Predict who won the Copley Medal
predict_object("Copley Medal", "winner")

# Predict the relation between two known entities
predict_relation("Ferdinand von Lindemann", "Martin Wilhelm Kutta")

# Predict doctoral students of a mathematician
predict_object("Felix Klein", "doctoral student")

In [ ]:
# How well represented is Felix Klein in the training set?
node2int = {v: k for k, v in wiki_graph.node_labels.items()}

# Define the target entity name
entity_name = "Felix Klein"

if entity_name not in node2int:
    print(f"'{entity_name}' not found in node labels. It might still be a Q-code.")
    matches = [n for n in node2int if entity_name.lower() in n.lower()]
    if matches:
        print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
    else:
        print("  No partial matches found.")
    print("  Perhaps try a Wikidata Q-code (e.g., 'Q12345') if you know it.")
else:
    klein_idx = node2int[entity_name]

    train_edges = wiki_graph.train_edges
    klein_edges = train_edges[train_edges[:, 0] == klein_idx]
    print(f"{entity_name} appears as subject in {len(klein_edges)} training edges")

    # Show what relations he has
    for edge in klein_edges:
        rel_label = wiki_graph.rel_labels.get(edge[2].item(), str(edge[2].item()))
        obj_label = wiki_graph.node_labels.get(edge[1].item(), str(edge[1].item()))
        print(f"  --[{rel_label}]--> {obj_label}")

In [ ]:
def entity_stats(name: str):
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}
    if name not in node2int:
        print(f"Entity '{name}' not found in node labels. It might still be a Q-code.")
        matches = [n for n in node2int if name.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):", ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        print("  Perhaps try a Wikidata Q-code (e.g., 'Q12345') if you know it.")
        return

    idx = node2int[name]
    as_subject = wiki_graph.train_edges[wiki_graph.train_edges[:, 0] == idx]
    as_object  = wiki_graph.train_edges[wiki_graph.train_edges[:, 1] == idx]
    print(f"'{name}' — {len(as_subject)} outgoing, {len(as_object)} incoming training edges")

In [ ]:
# 1. Find relations with enough data for reliable prediction
print("Relations with >1000 training edges:")
for r_idx in range(wiki_graph.num_relations):
    count = (wiki_graph.train_edges[:, 2] == r_idx).sum().item()
    if count > 1000:
        label = wiki_graph.rel_labels.get(r_idx, str(r_idx))
        print(f"  {label:<35} {count:>6,}")

In [ ]:
# See all valid relation types (only 283 — manageable to read)
for idx, label in sorted(wiki_graph.rel_labels.items(), key=lambda x: x[1]):
    print(label)